# GIVP Rust — Comparacao com a Literatura

Notebook para comparar **GIVP-full** vs **GRASP-only** na implementacao Rust em funcoes classicas.
A execucao acontece via programas Rust temporarios compilados com `cargo run`.

## 1. Setup

In [ ]:
"""Setup helpers for Rust literature-comparison benchmark notebook."""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

import pandas as pd


def find_cargo() -> str:
    """Return a usable cargo executable path."""
    found = shutil.which("cargo")
    if found:
        return found
    candidates = [
        Path.home() / ".cargo" / "bin" / "cargo.exe",
        Path.home() / ".cargo" / "bin" / "cargo",
    ]
    for c in candidates:
        if c.is_file():
            return str(c)
    raise RuntimeError("cargo nao encontrado")


CARGO = find_cargo()
ROOT = Path.cwd().resolve().parents[1]
RUST_DIR = ROOT / "rust"
WORK_DIR = ROOT / "Notebooks" / "_rust_lit_compare"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz: {ROOT}")
print(f"Rust crate: {RUST_DIR}")
print(f"Cargo: {CARGO}")

Raiz: D:\Projetos\Projeto_SOG2
Rust crate: D:\Projetos\Projeto_SOG2\rust
cargo 1.95.0 (f2d3ce0bd 2026-03-21)


## 2. Configuracao do experimento

In [2]:
N_RUNS = 30
DIMS = 10
MAX_ITERS = 100
OUT_JSON = Path("benchmark_literature_comparison_rust_results.json")

print(f"Rodadas: {N_RUNS}, dimensao: {DIMS}, max_iters: {MAX_ITERS}")

Rodadas: 30, dimensao: 10, max_iters: 100


## 3. Execucao (Rust)

In [ ]:
rust_template = textwrap.dedent(
    """
    use givp::{givp, Direction, GivpConfig};

    fn sphere(x: &[f64]) -> f64 { x.iter().map(|v| v * v).sum() }
    fn rosenbrock(x: &[f64]) -> f64 {
        x.windows(2)
            .map(|w| 100.0 * (w[1] - w[0].powi(2)).powi(2) + (1.0 - w[0]).powi(2))
            .sum()
    }
    fn rastrigin(x: &[f64]) -> f64 {
        let n = x.len() as f64;
        10.0 * n
            + x.iter()
                .map(|v| v * v - 10.0 * (2.0 * std::f64::consts::PI * v).cos())
                .sum::<f64>()
    }
    fn ackley(x: &[f64]) -> f64 {
        let n = x.len() as f64;
        let s1 = x.iter().map(|v| v * v).sum::<f64>() / n;
        let s2 = x.iter().map(|v| (2.0 * std::f64::consts::PI * v).cos()).sum::<f64>() / n;
        -20.0 * (-0.2 * s1.sqrt()).exp() - s2.exp() + 20.0 + std::f64::consts::E
    }

    fn cfg_full(seed: u64) -> GivpConfig {
        GivpConfig {
            seed: Some(seed),
            max_iterations: __MAX_ITERS__,
            alpha: 0.12,
            adaptive_alpha: true,
            alpha_min: 0.08,
            alpha_max: 0.18,
            vnd_iterations: 200,
            ils_iterations: 10,
            perturbation_strength: 4,
            use_elite_pool: true,
            elite_size: 7,
            path_relink_frequency: 8,
            use_cache: true,
            cache_size: 10_000,
            early_stop_threshold: 80,
            use_convergence_monitor: true,
            ..GivpConfig::default()
        }
    }

    fn cfg_grasp_only(seed: u64) -> GivpConfig {
        GivpConfig {
            seed: Some(seed),
            max_iterations: __MAX_ITERS__,
            alpha: 0.12,
            adaptive_alpha: false,
            vnd_iterations: 1,
            ils_iterations: 1,
            perturbation_strength: 0,
            use_elite_pool: false,
            use_convergence_monitor: false,
            use_cache: true,
            cache_size: 10_000,
            early_stop_threshold: __MAX_ITERS__,
            ..GivpConfig::default()
        }
    }

    fn main() {
        let funcs: Vec<(&str, fn(&[f64]) -> f64, (f64, f64))> = vec![
            ("Sphere", sphere, (-5.12, 5.12)),
            ("Rosenbrock", rosenbrock, (-5.0, 10.0)),
            ("Rastrigin", rastrigin, (-5.12, 5.12)),
            ("Ackley", ackley, (-32.768, 32.768)),
        ];

        let mut rows: Vec<String> = Vec::new();
        rows.push(String::from("["));

        for (algo_name, mk_cfg) in [
            ("GIVP-full", cfg_full as fn(u64) -> GivpConfig),
            ("GRASP-only", cfg_grasp_only as fn(u64) -> GivpConfig),
        ] {
            for (fname, f, b) in &funcs {
                let bounds = vec![*b; __DIMS__];
                for seed in 0..__N_RUNS__u64 {
                    let cfg = mk_cfg(seed);
                    let t0 = std::time::Instant::now();
                    let res = givp(*f, &bounds, cfg, Direction::Minimize).unwrap();
                    let dt = t0.elapsed().as_secs_f64();
                    let row = format!(
                        concat!(
                            "{\"algorithm\":\"{}\",",
                            "\"function\":\"{}\",",
                            "\"seed\":{},",
                            "\"fun\":{},",
                            "\"nfev\":{},",
                            "\"nit\":{},",
                            "\"time_s\":{}",
                            "},"
                        ),
                        algo_name, fname, seed, res.fun, res.nfev, res.nit, dt
                    );
                    rows.push(row);
                }
            }
        }

        if rows.len() > 1 {
            let last = rows.len() - 1;
            rows[last] = rows[last].trim_end_matches(',').to_string();
        }
        rows.push(String::from("]"));
        println!("{}", rows.join("\\n"));
    }
    """
)

rust_main = (
    rust_template.replace("__MAX_ITERS__", str(MAX_ITERS))
    .replace("__DIMS__", str(DIMS))
    .replace("__N_RUNS__", str(N_RUNS))
)

work = WORK_DIR / "lit_compare"
(work / "src").mkdir(parents=True, exist_ok=True)
crate_rel = os.path.relpath(str(RUST_DIR), str(work)).replace("\\", "/")
(work / "Cargo.toml").write_text(
    textwrap.dedent(
        f"""
        [package]
        name = "lit_compare_rust"
        version = "0.1.0"
        edition = "2021"

        [dependencies]
        givp = {{ path = "{crate_rel}" }}
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)
(work / "src" / "main.rs").write_text(rust_main, encoding="utf-8")

try:
    proc = subprocess.run(  # noqa: S603
        [CARGO, "run", "--quiet"],
        cwd=work,
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as err:
    raise RuntimeError(err.stderr or err.stdout or str(err)) from err

records = json.loads(proc.stdout)
df = pd.DataFrame(records)
print(f"Registros coletados: {len(df)}")
df.head()

SyntaxError: f-string: valid expression required before '}' (3700331801.py, line 80)

## 4. Estatisticas e teste de Wilcoxon

In [ ]:
from typing import Any

from scipy import stats

summary = (
    df.groupby(["function", "algorithm"])
    .agg(
        mean=("fun", "mean"),
        std=("fun", "std"),
        best=("fun", "min"),
        median=("fun", "median"),
        nfev=("nfev", "mean"),
        time_s=("time_s", "mean"),
    )
    .reset_index()
)
print(summary.round(6).to_string(index=False))

print("\nWilcoxon (GIVP-full < GRASP-only):")
for fname in sorted(df["function"].unique()):
    a = df[(df["function"] == fname) & (df["algorithm"] == "GIVP-full")][
        "fun"
    ].to_numpy()
    b = df[(df["function"] == fname) & (df["algorithm"] == "GRASP-only")][
        "fun"
    ].to_numpy()
    result: Any = stats.wilcoxon(a, b, alternative="less")
    stat_value = float(result.statistic)
    p_value = float(result.pvalue)
    sig = "SIM" if p_value < 0.05 else "NAO"
    print(f"- {fname:<12} W={stat_value:6.1f}  p={p_value:.4f}  significativo={sig}")

## 5. Exportar resultados

In [ ]:
payload = {
    "metadata": {
        "n_runs": N_RUNS,
        "dims": DIMS,
        "algorithms": ["GIVP-full", "GRASP-only"],
        "functions": sorted(df["function"].unique().tolist()),
    },
    "summary": json.loads(summary.to_json(orient="records")),
    "records": records,
}
OUT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Arquivo salvo: {OUT_JSON.resolve()}")